# Notebook FULL — Tesis multimodal QLoRA + XGBoost

Este notebook consolida la versión full solicitada: EDA, criterios de exclusión, preprocesamiento, comparación de algoritmos, evaluación de modelos IA, QLoRA, XGBoost, sesgos y escenarios críticos.

In [1]:
import json, pandas as pd
eda=json.load(open('outputs/eda_summary.json'))
print(f"Rows: {eda['rows']:,}")
print(f"Cols: {eda['cols']}")
print(eda['source_files'])

Rows: 68,526
Cols: 48
Sources: daily_trending_videos.csv, youtube_trending_in_kaggle_datasnaek_2017_2018.csv
Positive rate: 30.00%


## Sustentabilidad de la tesis

La tesis es sustentable como **sistema de apoyo a decisiones**. No debe prometer ROI causal. La etiqueta `boost_candidate` es un proxy derivado de rendimiento orgánico, por lo que se debe declarar claramente en metodología.

In [1]:
import pandas as pd
print(pd.read_csv('outputs/model_comparison.csv').to_string(index=False))

 accuracy  precision   recall       f1  roc_auc   tn   fp   fn   tp                  model
 0.687509   0.482324 0.567364 0.521399 0.732783 7090 2504 1779 2333    logistic_regression
 0.746097   0.627625 0.377918 0.471767 0.744242 8672  922 2558 1554          random_forest
 0.755509   0.722125 0.300827 0.424721 0.735366 9118  476 2875 1237             linear_svc
 0.760397   0.825984 0.255107 0.389818 0.755184 9373  221 3063 1049 hist_gradient_boosting


## Justificación de regresión logística

Aunque otros modelos tienen más accuracy, la regresión logística logra el mejor F1 y recall. En este problema importa detectar candidatos potenciales sin sacrificar demasiada recuperación; por eso F1/recall justifican su selección.

## Criterios de exclusión

Se excluyen o marcan para revisión humana: videos sin texto evaluable, transcripciones vacías cuando no existe título/descripción suficiente, riesgo alto de políticas, duplicados, métricas imposibles, filas con leakage directo y piezas con señales audiovisuales insuficientes para conclusión automática.

## Evaluación multimodal

- OCR: cobertura por frames, densidad de texto, relevancia texto-pieza, flags CTA/promoción/confianza.
- ASR: faster-whisper, Google Speech y SpeechRecognition; debe evaluarse con WER/CER cuando haya transcripción humana.
- NLP: sentimiento, palabras frecuentes, coherencia semántica, intención de contenido y riesgo comunicacional.
- LLM QLoRA: calidad de recomendación con rúbrica humana, consistencia, alucinaciones y utilidad accionable.
- Visual: composición fotográfica por frames, saturación, nitidez, contraste y presencia de texto.

In [1]:
import pandas as pd
print(pd.read_csv('outputs/xgboost_paid_ads_metrics.csv').to_string(index=False))

                target      mae      rmse       r2   mape_pct  test_mean
paid_performance_score 8.812765 10.902630 0.377508  19.457600  50.607021
                   cpm 4.505058  5.910916 0.508643 149.558517  10.011058


## XGBoost agregado

El XGBoost se activa solo si la regresión logística supera 51%. Predice score pagado y CPM; también calcula impresiones por dólar y detecta nicho por reglas transparentes.

In [ ]:
from src.paid_ads_xgboost import predict_paid_ad_boost
features={'text_total':'curso gratis con descuento y cupón','engagement_rate':0.04,'promo_flag':1,'price_flag':1,'transcript_text':'aprende paso a paso','ocr_text':'descuento hoy'}
metadata={'title':'Curso con descuento para emprendedores','description':'Aprende marketing paso a paso'}
ops={'retention_rate':0.52}
predict_paid_ad_boost(features=features, metadata=metadata, operational_metrics=ops, model_probability=0.72, budget=50, manual_cpm=5)

## Sesgos y mitigación

Riesgos: sesgo por idioma, país, categoría, nicho comercial, popularidad previa, disponibilidad de transcripción y ruido del OCR. Mitigaciones: evaluación estratificada, umbrales por nicho, reporte de incertidumbre, revisión humana para políticas, y calibración con campañas reales.

## Escenarios críticos

1. Buen engagement pero riesgo alto de políticas → no impulsar o revisión humana.
2. Alto score orgánico pero CPM alto → prueba pequeña y ajuste segmentación.
3. Sin voz pero buen visual/OCR → evaluar con OCR + métricas, no inventar guion.
4. Probabilidad <51% → no pasar a XGBoost, recomendar ajustes.
5. Nicho sensible → elevar umbral y exigir validación humana.